<a href="https://colab.research.google.com/github/DARIOYBETY123/Modulo-4-Deep-Leraning/blob/main/4_1_Descenso_Gradiente.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Descenso por Gradiente para Regresión Lineal Simple

cOmpute_loss

In [1]:
def compute_loss(data_x, data_y, model, phi):
    # Calcular predicciones del modelo usando data_x
    pred_y = model(phi, data_x)
    # Calcular la diferencia cuadrada y sumar todos los términos
    loss = np.sum((pred_y - data_y) ** 2)
    return loss

Con los parámetros de prueba phi0=0.6, phi1=-0.2 devuelve 12.367 (coincide con el valor esperado).

Cálculo del gradiente

In [3]:
def compute_gradient(data_x, data_y, phi):
    # Predicciones y errores
    pred_y = phi[0] + phi[1] * data_x
    errors = pred_y - data_y

    # Derivadas parciales
    dl_dphi0 = 2 * np.sum(errors)
    dl_dphi1 = 2 * np.sum(errors * data_x)

    # Devolver como vector columna
    return np.array([[dl_dphi0], [dl_dphi1]])

LA DERIVADA ES -10.769

In [4]:
# Importar librerías
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.colors import ListedColormap

# Datos de entrenamiento
data = np.array([[0.03,0.19,0.34,0.46,0.78,0.81,1.08,1.18,1.39,1.60,1.65,1.90],
                 [0.67,0.85,1.05,1.00,1.40,1.50,1.30,1.54,1.55,1.68,1.73,1.60]])

# Modelo lineal
def model(phi,x):
    y_pred = phi[0] + phi[1] * x
    return y_pred

# Dibujar modelo
def draw_model(data,model,phi,title=None):
    x_model = np.arange(0,2,0.01)
    y_model = model(phi,x_model)
    fig, ax = plt.subplots()
    ax.plot(data[0,:],data[1,:],'bo')
    ax.plot(x_model,y_model,'m-')
    ax.set_xlim([0,2]); ax.set_ylim([0,2])
    ax.set_xlabel('x'); ax.set_ylabel('y')
    ax.set_aspect('equal')
    if title is not None: ax.set_title(title)
    plt.show()

# ======================================
# SOLUCIÓN: Función de pérdida
# ======================================
def compute_loss(data_x, data_y, model, phi):
    pred_y = model(phi, data_x)
    errors = pred_y - data_y
    loss = np.sum(errors ** 2)
    return loss

# ======================================
# SOLUCIÓN: Gradiente
# ======================================
def compute_gradient(data_x, data_y, phi):
    pred_y = phi[0] + phi[1] * data_x
    errors = pred_y - data_y
    dl_dphi0 = 2 * np.sum(errors)
    dl_dphi1 = 2 * np.sum(errors * data_x)
    return np.array([[dl_dphi0], [dl_dphi1]])

# Funciones de búsqueda lineal (dadas, sin modificar)
def loss_function_1D(dist_prop, data, model, phi_start, search_direction):
    return compute_loss(data[0,:], data[1,:], model, phi_start - search_direction * dist_prop)

def line_search(data, model, phi, gradient, thresh=.00001, max_dist = 0.1, max_iter = 15, verbose=False):
    a, b, c, d = 0, 0.33 * max_dist, 0.66 * max_dist, 1.0 * max_dist
    n_iter = 0
    while np.abs(b-c) > thresh and n_iter < max_iter:
        n_iter += 1
        lossa = loss_function_1D(a, data, model, phi, gradient)
        lossb = loss_function_1D(b, data, model, phi, gradient)
        lossc = loss_function_1D(c, data, model, phi, gradient)
        lossd = loss_function_1D(d, data, model, phi, gradient)
        if np.argmin((lossa,lossb,lossc,lossd))==0:
            b, c, d = a+(b-a)/2, a+(c-a)/2, a+(d-a)/2
            continue
        if lossb < lossc:
            d = c
            b, c = a+(d-a)/3, a+2*(d-a)/3
            continue
        a = b
        b, c = a+(d-a)/3, a+2*(d-a)/3
    return (b+c)/2.0

# ======================================
# SOLUCIÓN: Paso de descenso
# ======================================
def gradient_descent_step(phi, data, model):
    grad = compute_gradient(data[0,:], data[1,:], phi)
    alpha = line_search(data, model, phi, grad)
    phi = phi - alpha * grad
    return phi

# -------------------
# Ejecutar todo
# -------------------
# Prueba de pérdida
phi_test = np.array([[0.6], [-0.2]])
loss = compute_loss(data[0,:],data[1,:],model,phi_test)
print(f'Pérdida calculada: {loss:.3f} | Esperada: 12.367')

# Prueba de gradiente vs diferencias finitas
grad = compute_gradient(data[0,:],data[1,:],phi_test)
delta = 0.0001
g0_est = (compute_loss(data[0,:],data[1,:],model,phi_test+[[delta],[0]]) - compute_loss(data[0,:],data[1,:],model,phi_test))/delta
g1_est = (compute_loss(data[0,:],data[1,:],model,phi_test+[[0],[delta]]) - compute_loss(data[0,:],data[1,:],model,phi_test))/delta
print(f'Gradiente analítico:  ({grad[0,0]:.4f}, {grad[1,0]:.4f})')
print(f'Gradiente numérico:   ({g0_est:.4f}, {g1_est:.4f})')

# Ejecutar descenso por gradiente
n_steps = 10
phi_all = np.zeros((2, n_steps+1))
phi_all[0,0], phi_all[1,0] = 1.6, -0.5

for step in range(n_steps):
    phi_all[:, step+1:step+2] = gradient_descent_step(phi_all[:, step:step+1], data, model)
    loss = compute_loss(data[0,:], data[1,:], model, phi_all[:, step+1:step+2])
    print(f'Iter {step+1:2d} | φ0={phi_all[0,step+1]:.4f}, φ1={phi_all[1,step+1]:.4f} | Pérdida: {loss:.5f}')

Pérdida calculada: 12.367 | Esperada: 12.367
Gradiente analítico:  (-21.9040, -26.8404)
Gradiente numérico:   (-21.9028, -26.8389)
Iter  1 | φ0=1.7082, φ1=-0.2034 | Pérdida: 2.83536
Iter  2 | φ0=1.2556, φ1=-0.0413 | Pérdida: 1.65686
Iter  3 | φ0=1.3138, φ1=0.1213 | Pérdida: 1.00551
Iter  4 | φ0=1.0663, φ1=0.2087 | Pérdida: 0.64911
Iter  5 | φ0=1.0980, φ1=0.2986 | Pérdida: 0.45046
Iter  6 | φ0=0.9601, φ1=0.3473 | Pérdida: 0.33967
Iter  7 | φ0=0.9778, φ1=0.3975 | Pérdida: 0.27787
Iter  8 | φ0=0.9007, φ1=0.4247 | Pérdida: 0.24337
Iter  9 | φ0=0.9106, φ1=0.4527 | Pérdida: 0.22413
Iter 10 | φ0=0.8677, φ1=0.4678 | Pérdida: 0.21340
